<a href="https://colab.research.google.com/github/rupeshk98/14/blob/main/defooocus_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

#@title DeFooocus
#@markdown **Launch the interface DeFocus (Fooocus fork)** | You need to connect with T4/A100/V100

print("[DeFooocus] Preparing ...")

theme = "dark" #@param ["dark", "light"]
preset = "deafult" #@param ["deafult", "realistic", "anime", "lcm", "sai", "turbo", "lighting", "hypersd", "playground_v2.5", "dpo", "spo", "sd1.5"]
advenced_args = "--share --attention-split --always-high-vram --disable-offload-from-vram --all-in-fp16" #@param {type: "string"}

if preset != "deafult":
  args = f"{advenced_args} --theme {theme} --preset {preset}"
else:
  args = f"{advenced_args} --theme {theme}"

# Fix for Python 3.12: Pin stable dependencies and bypass build errors
!pip install --upgrade pip
!pip install -q --prefer-binary "numpy==1.26.4" "gradio==3.48.0" "transformers==4.30.2" "tokenizers==0.13.3" "torchsde==0.2.6" pygit2 PyYAML>=6.0.2 rembg onnxruntime-gpu

%cd /content
if not os.path.exists('/content/DeFooocus'):
  !git clone https://github.com/rupeshk98/DeFooocus.git

%cd /content/DeFooocus

# Patch requirements to prevent version drift during setup
if os.path.exists('requirements_versions.txt'):
    !sed -i 's/==/>=/g' requirements_versions.txt
    !sed -i '/numpy/d' requirements_versions.txt
    !sed -i '/gradio/d' requirements_versions.txt
    !sed -i '/transformers/d' requirements_versions.txt
    !sed -i '/tokenizers/d' requirements_versions.txt

!pip install -q --prefer-binary -r requirements_versions.txt

# Robust Persistent Patching for compatibility issues
def apply_persistent_patches():
    # 1. Fix CLIP loading (no_init_weights)
    clip_patch = "modules/patch_clip.py"
    if os.path.exists(clip_patch):
        print(f"[DeFooocus] Patching {clip_patch} ...")
        with open(clip_patch, 'r') as f: lines = f.readlines()
        new_lines = ["import contextlib\n", "import torch\n"]
        for line in lines:
            if "with modeling_utils.no_init_weights():" in line:
                indent = line[:line.find("with")]
                new_lines.append(f"{indent}no_init = getattr(modeling_utils, 'no_init_weights', contextlib.nullcontext)\n")
                new_lines.append(f"{indent}with no_init():\n")
            else: new_lines.append(line)
        with open(clip_patch, 'w') as f: f.writelines(new_lines)

    # 2. Fix BLIP Describe/Interrogate (apply_chunking_to_forward)
    blip_patch = "extras/BLIP/models/med.py"
    if os.path.exists(blip_patch):
        print(f"[DeFooocus] Patching {blip_patch} for Describe feature...")
        with open(blip_patch, 'r') as f: content = f.read()
        # Safety for missing apply_chunking_to_forward
        content = content.replace("apply_chunking_to_forward,", "")
        content = "def apply_chunking_to_forward(a, b, c, d): return a(b, c, d)\n" + content
        with open(blip_patch, 'w') as f: f.write(content)

apply_persistent_patches()

print("[DeFooocus] Starting ...")
!python launch.py $args

[DeFooocus] Preparing ...
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
/content
/content/DeFooocus
[DeFooocus] Patching modules/patch_clip.py ...
[DeFooocus] Patching extras/BLIP/models/med.py for Describe feature...
[DeFooocus] Starting ...
[System ARGV] ['launch.py', '--share', '--attention-split', '--always-high-vram', '--disable-offload-from-vram', '--all-in-fp16', '--theme', 'dark']
Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Fooocus version: 0.2
Total VRAM 14913 MB, total RAM 12976 MB
Forcing FP16.
Set vram state to: HIGH_VRAM
Device: cuda:0 Tesla T4 : native
VAE dtype: torch.float32
Using 

### Save Files to GitHub
Use the cell below to push your generated images or code changes to a GitHub repository.
1. Go to your GitHub settings and create a **Personal Access Token (classic)** with `repo` permissions.
2. Add it to Colab Secrets as `GITHUB_TOKEN`.

In [ ]:
import os
from google.colab import userdata

# --- Configuration ---
GITHUB_USER = "your_username"  # @param {type:"string"}
GITHUB_REPO = "your_repo_name"  # @param {type:"string"}
GITHUB_EMAIL = "your_email@example.com"  # @param {type:"string"}
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except:
    print("Error: Please add 'GITHUB_TOKEN' to your Colab Secrets.")

# Configure Git Identity
!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "{GITHUB_USER}"

# Define the target folder to upload (e.g., DeFooocus outputs or a specific folder)
# To upload the whole DeFooocus folder, set upload_dir to '/content/DeFooocus'
upload_dir = "/content/DeFooocus/outputs" # @param {type:"string"}

if os.path.exists(upload_dir):
    %cd {upload_dir}
    if not os.path.exists('.git'):
        !git init
        !git remote add origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git

    !git add .
    !git commit -m "Update from Colab: DeFooocus outputs"
    !git branch -M main
    !git push -u origin main
else:
    print(f"Directory {upload_dir} not found.")

### ⚠️ Troubleshooting "An error occurred" when saving to GitHub

If you see an error when using **File > Save a copy in GitHub**, it is usually because you are trying to save to the original developer's repository.

**To fix it:**
1. Create a **new, empty repository** on your own GitHub account.
2. In Colab, click **File > Save a copy in GitHub**.
3. In the **Repository** dropdown, make sure you select **[Your-Username]/[Your-New-Repo]** instead of the original `rupeshk98` one.
4. Click **OK**.